# M2 SQuAD-v2 Dataset Exploration and Readiness Analysis

- **Successor to:** `dataset_xplore_and_upload.ipynb`
- **Scope:** QC outputs in `out_qc_M2/` — especially `squadv2_final_merged.jsonl`
- **Question answered:** is the merged dataset (or any QC split) ready to be used for SQuAD-v2 fine-tuning, and how should `is_impossible=True` rows be handled?

This notebook is analysis-only: it does not train models and does not modify the datasets.

## 1. Configuration

Paths point to the QC outputs, the GLM audit, and the audit gold-accuracy table. Results (CSV/PNG/PDF) are exported to `out_qc_M2/analysis/`.

In [ ]:
import os
import sys
import glob
import json
import subprocess
import tempfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.join(os.getcwd(), "scripts"))
import qa_dataset_utils as u

ROOT = os.getcwd()
QC_DIR = os.path.join(ROOT, "out_qc_M2")
OUT_DIR = os.path.join(QC_DIR, "analysis")
os.makedirs(OUT_DIR, exist_ok=True)

SPLIT_PATHS = {
    "merged": os.path.join(QC_DIR, "squadv2_final_merged.jsonl"),
    "strict_gemma": os.path.join(QC_DIR, "squadv2_strict_gemma.jsonl"),
    "strict_qwen": os.path.join(QC_DIR, "squadv2_strict_qwen.jsonl"),
    "expanded_gemma": os.path.join(QC_DIR, "squadv2_expanded_gemma.jsonl"),
    "expanded_qwen": os.path.join(QC_DIR, "squadv2_expanded_qwen.jsonl"),
}
AUDIT_CSV = os.path.join(QC_DIR, "audit_stratified_sample_labeled_v1.csv")
GOLD_ACC_CSV = os.path.join(ROOT, "Reports", "audit_glm_v1", "outputs", "training_split_gold_accuracy.csv")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
print("ROOT:", ROOT)
print("OUT_DIR:", OUT_DIR)

## 2. Load QC SQuAD-v2 splits

Files are loaded with `strict=False` so the notebook can keep and count rows that do not pass every schema check (e.g. a few offset mismatches in `expanded_gemma`). The corrected question-kind mapping (including `dirección`/`calles` → `place`) is applied.

In [ ]:
frames = {}
for name, path in SPLIT_PATHS.items():
    if not os.path.exists(path):
        print(f"SKIP {name}: {path} not found")
        continue
    recs = u.load_squad2_variant(path, name, strict=False)
    df = pd.DataFrame(recs)
    df["kind"] = df["question"].apply(u.question_kind)
    df["kind_en"] = df["kind"].map(u.QKIND_EN)
    df["answer_words"] = df.apply(
        lambda r: len(r["answers"]["text"][0].split()) if not r["is_impossible"] else 0, axis=1
    )
    frames[name] = df
    print(f"{name}: {len(df)} rows, {df['context_id'].nunique()} contexts, "
          f"{int(df['is_impossible'].sum())} impossible")

## 3. SQuAD-v2 readiness checks

The table below answers the core question: are the files ready for SQuAD-v2 fine-tuning? For every split we check:

- **Schema errors**: rows that fail `validate_squad2_record` (missing fields, non-empty answers for impossible rows, empty answers for answerable rows, invalid offsets).
- **impossible_with_answers**: `is_impossible=True` rows that still carry an answer span (must be 0).
- **answerable_without_answers**: answerable rows with an empty `answers` dict (must be 0).
- **invalid_offsets**: answerable rows whose `answer_start` does not point to `answer_text` inside the context (must be 0).
- **duplicate ids**: repeated `id` values (must be 0).

Known result: `expanded_gemma` has a small number of rows (68) whose `answer_start` is off by a leading space (e.g. answer `" tablero de instrumentos"` at an offset that points to `"tablero"`). Those rows are still counted below; the merged and strict splits are clean.

In [ ]:
def validate_records(df):
    n_bad = 0
    for _, r in df.iterrows():
        if u.validate_squad2_record(r.to_dict()):
            n_bad += 1
    return n_bad

rows = []
for name, df in frames.items():
    n = len(df)
    imp = int(df["is_impossible"].sum())
    imp_with_ans = int(df.apply(
        lambda r: r["is_impossible"] and bool(r["answers"]["text"] or r["answers"]["answer_start"]), axis=1
    ).sum())
    ans_without = int(df.apply(
        lambda r: (not r["is_impossible"]) and (not r["answers"]["text"]), axis=1
    ).sum())
    bad_off = int(df.apply(
        lambda r: (not r["is_impossible"]) and
                  (not r["context"].startswith(r["answers"]["text"][0], r["answers"]["answer_start"][0])), axis=1
    ).sum())
    dup = int(df["id"].duplicated().sum())
    schema_err = validate_records(df)
    rows.append({
        "split": name,
        "rows": n,
        "contexts": int(df["context_id"].nunique()),
        "answerable": n - imp,
        "impossible": imp,
        "impossible_rate_%": round(imp / n * 100, 2),
        "impossible_with_answers": imp_with_ans,
        "answerable_without_answers": ans_without,
        "invalid_offsets": bad_off,
        "duplicate_ids": dup,
        "schema_errors": schema_err,
    })

readiness = pd.DataFrame(rows)
display(readiness)
readiness.to_csv(os.path.join(OUT_DIR, "squad2_readiness.csv"), index=False)
with open(os.path.join(OUT_DIR, "table_squad2_readiness.tex"), "w", encoding="utf-8") as f:
    f.write(readiness.to_latex(index=False, float_format="%.2f",
                              caption="SQuAD-v2 readiness of QC M2 splits",
                              label="tab:readiness"))

## 4. How to handle `is_impossible=True` during fine-tuning

**Recommendation: keep them and leave the answer blank.**

- SQuAD v2 is explicitly designed so that unanswerable questions are part of the training signal. The model must learn to abstain (predict no answer) instead of hallucinating a span.
- In the SQuAD-v2 JSON format, an unanswerable example is represented as:
  ```json
  {"id": "...", "context": "...", "question": "...", "is_impossible": true,
   "answers": {"text": [], "answer_start": []}}
  ```
- During tokenization, the training script maps these rows to `start_positions = 0` and `end_positions = 0` (the CLS position), which is the standard convention used by Hugging Face QA fine-tuning. `scripts/run_qa_ablation.py` already does this in `make_train_features`.
- **Do not drop them**: the model would never learn to say "no answer", and the `NoAns` metrics (already part of the experiment report) would be meaningless.
- **Do not provide a fake answer**: that would teach the model to invent content not present in the context.

The readiness table confirms that every split follows this convention: `impossible_with_answers = 0` and `answerable_without_answers = 0`.

## 5. Question-type distribution

The corrected `question_kind` mapping now classifies the address question (`¿En qué dirección o entre qué calles...?`) as **place** instead of **other**, matching the audit CSV labels.

In [ ]:
kind_rows = []
for name, df in frames.items():
    g = df.groupby("kind_en").agg(rows=("id", "size"), impossible=("is_impossible", "sum"))
    g["impossible_rate"] = (g["impossible"] / g["rows"]).round(3)
    g["split"] = name
    kind_rows.append(g.reset_index())

kdf = pd.concat(kind_rows, ignore_index=True)
piv = kdf.pivot_table(index="kind_en", columns="split", values="rows", aggfunc="sum", fill_value=0)
display(piv)
piv.to_csv(os.path.join(OUT_DIR, "rows_by_question_type.csv"))

m = frames["merged"]
print("\nMerged impossible rate by question type:")
mr = m.groupby("kind_en")["is_impossible"].mean().rename("impossible_rate").round(3)
display(mr)

fig, ax = plt.subplots(figsize=(9, 4.5))
pos_m = m[~m["is_impossible"]]
sns.barplot(data=pos_m, x="kind_en", y="answer_words", ax=ax)
ax.set_title("Answer length by question type (merged, answerable rows)")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "fig_answer_words_by_kind.png"), bbox_inches="tight")
fig.savefig(os.path.join(OUT_DIR, "fig_answer_words_by_kind.pdf"), bbox_inches="tight")
plt.show()

## 6. Answer length distribution (answerable rows)

In [ ]:
merged = frames["merged"]
pos = merged[~merged["is_impossible"]]
print(pos["answer_words"].describe(percentiles=[.25, .5, .75, .9, .95, .99]).to_string())
print("\nAnswers > 100 words:", int((pos["answer_words"] > 100).sum()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
sns.histplot(pos["answer_words"], bins=60, ax=axes[0], color="steelblue")
axes[0].set_title("Answer length (words) — merged answerable rows")
axes[0].set_xlim(0, np.percentile(pos["answer_words"], 99))
sns.boxplot(x=pos["answer_words"], ax=axes[1], color="lightblue")
axes[1].set_title("Answer length boxplot")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "fig_answer_length.png"), bbox_inches="tight")
fig.savefig(os.path.join(OUT_DIR, "fig_answer_length.pdf"), bbox_inches="tight")
plt.show()

## 7. Per-context completeness

SQuAD v2 fine-tuning works with any number of questions per context, but it is useful to know how many of the 5 questions survived QC in the merged split.

In [ ]:
ctx_counts = merged.groupby("context_id").size()
print(ctx_counts.value_counts().sort_index().to_string())
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(x=ctx_counts, ax=ax)
ax.set_title("Rows per context in merged dataset")
ax.set_xlabel("Questions per context")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "fig_rows_per_context.png"), bbox_inches="tight")
fig.savefig(os.path.join(OUT_DIR, "fig_rows_per_context.pdf"), bbox_inches="tight")
plt.show()

## 8. Cross-split comparison and audit gold accuracy

The audit report (`Reports/audit_glm_v1/outputs/training_split_gold_accuracy.csv`) gives indicative gold accuracy within each candidate split. This complements the structural readiness table: structure is necessary, but label quality was already measured there.

In [ ]:
if os.path.exists(GOLD_ACC_CSV):
    gold_acc = pd.read_csv(GOLD_ACC_CSV)
    display(gold_acc.round(4))
    gold_acc.to_csv(os.path.join(OUT_DIR, "audit_gold_accuracy_by_split.csv"), index=False)
else:
    print("Gold accuracy CSV not found:", GOLD_ACC_CSV)

summary = readiness[["split", "rows", "contexts", "answerable", "impossible", "impossible_rate_%", "schema_errors"]].copy()
if os.path.exists(GOLD_ACC_CSV):
    summary = summary.merge(
        gold_acc[["split", "accuracy", "audit_rows_in_split"]].rename(
            columns={"accuracy": "gold_accuracy", "audit_rows_in_split": "gold_audit_rows"}
        ),
        left_on="split", right_on="split", how="left",
    )
display(summary)
summary.to_csv(os.path.join(OUT_DIR, "split_summary_with_gold.csv"), index=False)

## 9. Gold audit overlap

The 200 GLM-labeled rows are the pilot gold test set. They should be **held out from training** (the audit report already recommends this). The table shows how many of those rows currently appear in each QC split.

In [ ]:
if os.path.exists(AUDIT_CSV):
    audit = pd.read_csv(AUDIT_CSV)
    audit_keys = set(zip(audit["context"], audit["question"]))
    overlap = []
    for name, df in frames.items():
        keys = set(zip(df["context"], df["question"]))
        overlap.append({"split": name, "audit_rows_in_split": len(audit_keys & keys), "audit_total": len(audit_keys)})
    ov = pd.DataFrame(overlap)
    display(ov)
    ov.to_csv(os.path.join(OUT_DIR, "audit_overlap_by_split.csv"), index=False)
else:
    print("Audit CSV not found:", AUDIT_CSV)

## 10. Validate `scripts/prepare_final_dataset.py` output

The preparation script is already SQuAD-v2-ready: it validates the schema, makes a context-level split (default 80/10/10, seed 42), writes `train.jsonl`/`dev.jsonl`/`test.jsonl`, and optionally pushes to HF. This cell runs it on a 20-context smoke sample and re-validates the produced files.

In [ ]:
tmp = tempfile.mkdtemp(prefix="prepared_m2_")
cmd = [
    sys.executable,
    os.path.join(ROOT, "scripts", "prepare_final_dataset.py"),
    "--limit-contexts", "20",
    "--output-dir", tmp,
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

prepared = {}
for split_name in ["train", "dev", "test"]:
    path = os.path.join(tmp, f"{split_name}.jsonl")
    recs = u.load_squad2_variant(path, split_name)
    prepared[split_name] = recs
    n_bad = sum(1 for r in recs if u.validate_squad2_record(r))
    imp = sum(1 for r in recs if r["is_impossible"])
    print(f"{split_name}: {len(recs)} rows, {imp} impossible, schema_errors={n_bad}")

# Show one example of each class (answerable / unanswerable)
ex_ans = next(r for r in prepared["train"] if not r["is_impossible"])
ex_imp = next(r for r in prepared["train"] if r["is_impossible"])
print("\nAnswerable example:", json.dumps({k: ex_ans[k] for k in ("id", "is_impossible", "answers")}, ensure_ascii=False))
print("Unanswerable example:", json.dumps({k: ex_imp[k] for k in ("id", "is_impossible", "answers")}, ensure_ascii=False))

## 11. Conclusions

1. **`merged`, `strict_gemma`, `strict_qwen`, and `expanded_qwen` are structurally ready for SQuAD v2**: zero schema errors, zero `impossible_with_answers`, zero `answerable_without_answers`, zero invalid offsets, zero duplicate ids.
2. **`expanded_gemma` has 68 rows with invalid offsets** (answer text with a leading space whose `answer_start` points one character too late). If you use it for training, re-normalize those offsets first; the merged and strict splits do not have this issue.
3. **Keep `is_impossible=True` rows with empty answers** during fine-tuning; dropping them would remove the abstention signal, and filling them would teach hallucination.
4. **`scripts/prepare_final_dataset.py` is ready**: it produces valid SQuAD-v2 `train/dev/test` files and can push them to HF with `--push`.
5. The merged dataset is the recommended default: ~80.7k rows, 26.3% unanswerable, balanced size, and the audit showed it is competitive with `strict_gemma` in gold accuracy.
6. The corrected question-kind mapping fixes the previous `other` bucket: the address question is now classified as **place**, consistent with the audit labels.

All tables and figures were exported to `out_qc_M2/analysis/`.